# SVM + DeepFool Dual-Stream Consistency Gate Pipeline

This notebook mirrors the LR and NN DeepFool consistency-gate workflow for an SVM classifier.

Important implementation note:

- DeepFool needs input gradients.
- A standard sklearn SVM does not provide those gradients.
- This notebook trains a small TensorFlow surrogate only to generate DeepFool adversarial samples.
- The actual clean model, adversarially trained guardian model, evaluations, and consistency gate are all SVM-based.

Pipeline steps:

1. Train a clean SVM on clean training data
2. Train a small TensorFlow surrogate and generate DeepFool adversarial samples
3. Evaluate the clean SVM on clean, adversarial, and combined test data
4. Train a guardian SVM on clean plus DeepFool samples
5. Evaluate the guardian SVM
6. Run the dual-stream consistency gate


In [15]:
# If needed, install once:
# !pip install tensorflow adversarial-robustness-toolbox scikit-learn pandas numpy joblib

import warnings
warnings.filterwarnings("ignore")

import ast
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf

from art.attacks.evasion import DeepFool
from art.estimators.classification import TensorFlowV2Classifier

try:
    from IPython.display import display
except ImportError:
    display = print


In [16]:
# -----------------------------
# Configuration
# -----------------------------
def normalize_path(path_like) -> Path:
    """Make notebook paths work on Windows, macOS, Linux, and VS Code/Jupyter."""
    return Path(str(path_like).replace("\\", "/")).expanduser()


def resolve_existing_path(*candidates, required: bool = True):
    checked = []
    for candidate in candidates:
        if candidate is None:
            continue
        p = normalize_path(candidate)
        variants = [p]
        if not p.is_absolute():
            variants.extend([
                Path.cwd() / p,
                Path.cwd().parent / p,
                Path.cwd().parent.parent / p,
            ])
        for variant in variants:
            checked.append(str(variant))
            if variant.exists():
                print(f"Resolved path: {variant}")
                return variant
    if required:
        raise FileNotFoundError("Could not find any of these paths:\n" + "\n".join(checked))
    return None

DATA_PATH = resolve_existing_path(
    Path("../../CSVs/dataset.csv"),
    Path("../CSVs/dataset.csv"),
    Path("CSVs/dataset.csv"),
)

LABEL_COL = "anomaly"

# Drop non-feature columns if needed
DROP_COLS = {LABEL_COL, "segment", "train", "sampling", "channel"}

TEST_SIZE = 0.2
SEED = 42

# Surrogate model settings used only for generating DeepFool samples.
SURROGATE_HIDDEN = 64
SURROGATE_LR = 1e-3
SURROGATE_BATCH_SIZE = 128
SURROGATE_EPOCHS = 20

# DeepFool settings
DEEPFOOL_MAX_ITER = 50
DEEPFOOL_EPSILON = 1e-6
DEEPFOOL_NB_GRADS = 2
DEEPFOOL_BATCH_SIZE = SURROGATE_BATCH_SIZE

# For sklearn models, adversarial training is approximated by adding
# DeepFool samples generated from the surrogate model back into the training set.
ADV_RATIO = 0.85

# Save paths
SAVE_MODELS = True
ARTIFACT_DIR = Path("artifacts")
RESULTS_DIR = Path("results")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# SVM settings
BEST_PARAMS_PATH = resolve_existing_path(
    Path("../../MachineLearning/SVM/best_params.csv"),
    Path("../MachineLearning/SVM/best_params.csv"),
    Path("MachineLearning/SVM/best_params.csv"),
    required=False,
)

DEFAULT_SVM_PARAMS = {
    "C": 1.0,
    "kernel": "rbf",
    "gamma": "scale",
    "degree": 3,
    "class_weight": "balanced",
    "probability": True,
    "random_state": SEED,
}


Resolved path: ..\..\CSVs\dataset.csv
Resolved path: ..\..\MachineLearning\SVM\best_params.csv


In [17]:
from sklearn.svm import SVC

def coerce_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        text = value.strip()
        if text.lower() in {"none", "nan", ""}:
            return None
        if text.lower() in {"true", "false"}:
            return text.lower() == "true"
        try:
            return ast.literal_eval(text)
        except Exception:
            return text
    return value


def load_best_params(path: Path, defaults: dict):
    params = defaults.copy()

    if path is not None and path.exists():
        best_df = pd.read_csv(path)
        row = best_df.iloc[0].to_dict()
        for key, value in row.items():
            if key in params:
                params[key] = coerce_value(value)
        print(f"Loaded SVM params from {path}")
    else:
        print(f"Best params file not found at {path}. Using DEFAULT_SVM_PARAMS.")

    # Required for probability-based consistency gate.
    params["probability"] = True
    params["random_state"] = SEED
    if params.get("class_weight") is None:
        params["class_weight"] = "balanced"

    return params


SVM_PARAMS = load_best_params(BEST_PARAMS_PATH, DEFAULT_SVM_PARAMS)
print("SVM_PARAMS:", SVM_PARAMS)


def make_svm_model():
    return SVC(**SVM_PARAMS)


Loaded SVM params from ..\..\MachineLearning\SVM\best_params.csv
SVM_PARAMS: {'C': 1, 'kernel': 'linear', 'gamma': 'scale', 'degree': 3, 'class_weight': 'balanced', 'probability': True, 'random_state': 42}


In [18]:
def load_and_prepare(csv_path: str):
    df = pd.read_csv(csv_path)

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in {csv_path}.")

    y = df[LABEL_COL].astype(int).to_numpy()
    feature_df = df[[c for c in df.columns if c not in DROP_COLS]].copy()

    non_numeric_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric_cols:
        raise ValueError(
            f"Non-numeric feature columns found in {csv_path}: {non_numeric_cols}. "
            "Add them to DROP_COLS or encode them before training."
        )

    feature_cols = feature_df.columns.tolist()
    X = feature_df.to_numpy(dtype=np.float32)

    if len(np.unique(y)) != 2:
        raise ValueError(f"Expected binary labels, got: {np.unique(y)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    print(f"Loaded: {csv_path}")
    print(f"Rows={len(df)}, Features={X.shape[1]}, Label dist={np.bincount(y)}")
    print(f"Train={X_train.shape}, Test={X_test.shape}")

    return X_train, X_test, y_train.astype(np.int64), y_test.astype(np.int64), scaler, feature_cols


def make_balanced_training_set(X: np.ndarray, y: np.ndarray, seed: int = SEED):
    """Randomly oversample the minority class so sklearn models do not collapse to only class 0."""
    y_int = y.astype(int)
    classes, counts = np.unique(y_int, return_counts=True)
    if len(classes) != 2:
        raise ValueError(f"Expected two classes for balancing, got {classes}")
    max_count = int(counts.max())
    rng = np.random.default_rng(seed)
    sampled_indices = []
    for cls in classes:
        cls_indices = np.flatnonzero(y_int == cls)
        sampled_indices.append(rng.choice(cls_indices, size=max_count, replace=len(cls_indices) < max_count))
    sampled_indices = np.concatenate(sampled_indices)
    rng.shuffle(sampled_indices)
    X_balanced = X[sampled_indices].astype(np.float32)
    y_balanced = y_int[sampled_indices].astype(np.int64)
    print("Original train label dist:", dict(zip(classes.tolist(), counts.tolist())))
    print("Balanced train label dist:", dict(zip(*np.unique(y_balanced, return_counts=True))))
    return X_balanced, y_balanced

X_train, X_test, y_train, y_test, scaler, feature_cols = load_and_prepare(str(DATA_PATH))
X_train_balanced, y_train_balanced = make_balanced_training_set(X_train, y_train)


Loaded: ..\..\CSVs\dataset.csv
Rows=2123, Features=18, Label dist=[1689  434]
Train=(1698, 18), Test=(425, 18)
Original train label dist: {0: 1351, 1: 347}
Balanced train label dist: {np.int64(0): np.int64(1351), np.int64(1): np.int64(1351)}


In [19]:
def build_surrogate_model(d_in: int, hidden: int = SURROGATE_HIDDEN):
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(d_in,)),
        tf.keras.layers.Dense(hidden, activation="relu"),
        tf.keras.layers.Dense(hidden, activation="relu"),
        tf.keras.layers.Dense(2, activation="softmax"),
    ])


def make_surrogate_art_classifier(d_in: int):
    model = build_surrogate_model(d_in=d_in)
    optimizer = tf.keras.optimizers.Adam(learning_rate=SURROGATE_LR)
    loss_object = tf.keras.losses.CategoricalCrossentropy()

    @tf.function
    def train_step(model_instance, x_batch, y_batch):
        with tf.GradientTape() as tape:
            predictions = model_instance(x_batch, training=True)
            loss = loss_object(y_batch, predictions)
        gradients = tape.gradient(loss, model_instance.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model_instance.trainable_variables))

    return TensorFlowV2Classifier(
        model=model,
        nb_classes=2,
        input_shape=(d_in,),
        loss_object=loss_object,
        train_step=train_step,
        clip_values=(0.0, 1.0),
    )


def to_one_hot(y: np.ndarray, num_classes: int = 2):
    return tf.keras.utils.to_categorical(y.astype(int), num_classes=num_classes).astype(np.float32)


def predict_labels(model, X: np.ndarray):
    return model.predict(X).astype(int)


def predict_proba(model, X: np.ndarray):
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X)
    else:
        preds = model.predict(X)
        probs = np.zeros((len(preds), 2), dtype=np.float32)
        probs[np.arange(len(preds)), preds.astype(int)] = 1.0
    return probs.astype(np.float32)


def eval_sklearn_classifier(model, X: np.ndarray, y_true: np.ndarray, name: str):
    y_pred = predict_labels(model, X)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }, y_pred


def make_adv_training_set(X_clean, y_clean, X_adv, y_adv, adv_ratio: float = ADV_RATIO):
    n_adv = max(1, int(round(len(X_clean) * adv_ratio)))
    n_adv = min(n_adv, len(X_adv))

    rng = np.random.default_rng(SEED)
    adv_idx = rng.choice(len(X_adv), size=n_adv, replace=False)

    X_mix = np.vstack([X_clean, X_adv[adv_idx]]).astype(np.float32)
    y_mix = np.concatenate([y_clean, y_adv[adv_idx]]).astype(np.int64)

    shuffle_idx = rng.permutation(len(X_mix))
    return X_mix[shuffle_idx], y_mix[shuffle_idx]


In [20]:
# Step 1: train clean SVM model
print("Training clean SVM with:", SVM_PARAMS)

clean_model = make_svm_model()
clean_model.fit(X_train_balanced, y_train_balanced)

clean_on_clean, y_pred_clean = eval_sklearn_classifier(
    clean_model,
    X_test,
    y_test,
    "clean_model_on_clean_test",
)

if SAVE_MODELS:
    joblib.dump(clean_model, ARTIFACT_DIR / "svm_clean_model.joblib")
    joblib.dump(scaler, ARTIFACT_DIR / "svm_scaler.joblib")


Training clean SVM with: {'C': 1, 'kernel': 'linear', 'gamma': 'scale', 'degree': 3, 'class_weight': 'balanced', 'probability': True, 'random_state': 42}

[clean_model_on_clean_test] acc=0.8988 f1=0.7485
confusion matrix:
[[318  20]
 [ 23  64]]
              precision    recall  f1-score   support

           0     0.9326    0.9408    0.9367       338
           1     0.7619    0.7356    0.7485        87

    accuracy                         0.8988       425
   macro avg     0.8472    0.8382    0.8426       425
weighted avg     0.8976    0.8988    0.8982       425



In [21]:
# Step 2: train TensorFlow surrogate and generate DeepFool adversarial samples
print("Training TensorFlow surrogate used only for DeepFool generation with:", {
    "hidden": SURROGATE_HIDDEN,
    "learning_rate": SURROGATE_LR,
    "batch_size": SURROGATE_BATCH_SIZE,
    "epochs": SURROGATE_EPOCHS,
    "deepfool_max_iter": DEEPFOOL_MAX_ITER,
    "deepfool_epsilon": DEEPFOOL_EPSILON,
    "deepfool_nb_grads": DEEPFOOL_NB_GRADS,
})

surrogate_art = make_surrogate_art_classifier(d_in=X_train.shape[1])
surrogate_art.fit(
    X_train_balanced,
    to_one_hot(y_train_balanced, 2),
    batch_size=SURROGATE_BATCH_SIZE,
    nb_epochs=SURROGATE_EPOCHS,
)

def make_deepfool_attack(classifier):
    try:
        return DeepFool(
            classifier=classifier,
            max_iter=DEEPFOOL_MAX_ITER,
            epsilon=DEEPFOOL_EPSILON,
            nb_grads=DEEPFOOL_NB_GRADS,
            batch_size=DEEPFOOL_BATCH_SIZE,
            verbose=False,
        )
    except TypeError:
        return DeepFool(
            estimator=classifier,
            max_iter=DEEPFOOL_MAX_ITER,
            epsilon=DEEPFOOL_EPSILON,
            nb_grads=DEEPFOOL_NB_GRADS,
            batch_size=DEEPFOOL_BATCH_SIZE,
            verbose=False,
        )

deepfool = make_deepfool_attack(surrogate_art)

X_train_adv = np.clip(deepfool.generate(x=X_train), 0.0, 1.0).astype(np.float32)
X_test_adv = np.clip(deepfool.generate(x=X_test), 0.0, 1.0).astype(np.float32)

print("Adversarial data generated with DeepFool:")
print("X_train_adv:", X_train_adv.shape)
print("X_test_adv:", X_test_adv.shape)

# Keep adversarial labels aligned with original ground-truth labels.
y_train_adv = y_train.copy()
y_test_adv = y_test.copy()

X_test_combined = np.vstack([X_test, X_test_adv]).astype(np.float32)
y_test_combined = np.concatenate([y_test, y_test_adv]).astype(np.int64)


Training TensorFlow surrogate used only for DeepFool generation with: {'hidden': 64, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 20, 'deepfool_max_iter': 50, 'deepfool_epsilon': 1e-06, 'deepfool_nb_grads': 2}
Adversarial data generated with DeepFool:
X_train_adv: (1698, 18)
X_test_adv: (425, 18)


In [22]:
# Evaluate clean model on adversarial and combined test data
clean_on_adv, y_pred_adv_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_adv,
    y_test_adv,
    "clean_model_on_adv_test",
)

clean_on_combined, y_pred_combined_clean_model = eval_sklearn_classifier(
    clean_model,
    X_test_combined,
    y_test_combined,
    "clean_model_on_combined_test",
)



[clean_model_on_adv_test] acc=0.3224 f1=0.0526
confusion matrix:
[[129 209]
 [ 79   8]]
              precision    recall  f1-score   support

           0     0.6202    0.3817    0.4725       338
           1     0.0369    0.0920    0.0526        87

    accuracy                         0.3224       425
   macro avg     0.3285    0.2368    0.2626       425
weighted avg     0.5008    0.3224    0.3866       425


[clean_model_on_combined_test] acc=0.6106 f1=0.3032
confusion matrix:
[[447 229]
 [102  72]]
              precision    recall  f1-score   support

           0     0.8142    0.6612    0.7298       676
           1     0.2392    0.4138    0.3032       174

    accuracy                         0.6106       850
   macro avg     0.5267    0.5375    0.5165       850
weighted avg     0.6965    0.6106    0.6425       850



In [23]:
# Step 3: adversarial training approximation for SVM
X_train_mixed, y_train_mixed = make_adv_training_set(
    X_train,
    y_train,
    X_train_adv,
    y_train_adv,
    adv_ratio=ADV_RATIO,
)

print("Training guardian SVM with clean + DeepFool samples:")
print("X_train_mixed:", X_train_mixed.shape)
print("y_train_mixed:", y_train_mixed.shape)

X_train_mixed_balanced, y_train_mixed_balanced = make_balanced_training_set(X_train_mixed, y_train_mixed, seed=SEED + 1)
print("X_train_mixed_balanced:", X_train_mixed_balanced.shape)
print("y_train_mixed_balanced:", y_train_mixed_balanced.shape)

adv_model = make_svm_model()
adv_model.fit(X_train_mixed_balanced, y_train_mixed_balanced)

if SAVE_MODELS:
    joblib.dump(adv_model, ARTIFACT_DIR / "svm_adversarial_trained_model.joblib")


Training guardian SVM with clean + DeepFool samples:
X_train_mixed: (3141, 18)
y_train_mixed: (3141,)
Original train label dist: {0: 2499, 1: 642}
Balanced train label dist: {np.int64(0): np.int64(2499), np.int64(1): np.int64(2499)}
X_train_mixed_balanced: (4998, 18)
y_train_mixed_balanced: (4998,)


In [24]:
# Step 4: evaluate adversarially trained guardian model
adv_trained_on_adv, y_pred_adv = eval_sklearn_classifier(
    adv_model,
    X_test_adv,
    y_test_adv,
    "adv_trained_model_on_adv_test",
)

adv_trained_on_clean, y_pred_clean_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test,
    y_test,
    "adv_trained_model_on_clean_test",
)

adv_trained_on_combined, y_pred_combined_adv_model = eval_sklearn_classifier(
    adv_model,
    X_test_combined,
    y_test_combined,
    "adv_trained_model_on_combined_test",
)



[adv_trained_model_on_adv_test] acc=0.8776 f1=0.6829
confusion matrix:
[[317  21]
 [ 31  56]]
              precision    recall  f1-score   support

           0     0.9109    0.9379    0.9242       338
           1     0.7273    0.6437    0.6829        87

    accuracy                         0.8776       425
   macro avg     0.8191    0.7908    0.8036       425
weighted avg     0.8733    0.8776    0.8748       425


[adv_trained_model_on_clean_test] acc=0.8471 f1=0.6012
confusion matrix:
[[311  27]
 [ 38  49]]
              precision    recall  f1-score   support

           0     0.8911    0.9201    0.9054       338
           1     0.6447    0.5632    0.6012        87

    accuracy                         0.8471       425
   macro avg     0.7679    0.7417    0.7533       425
weighted avg     0.8407    0.8471    0.8431       425


[adv_trained_model_on_combined_test] acc=0.8624 f1=0.6422
confusion matrix:
[[628  48]
 [ 69 105]]
              precision    recall  f1-score   support


In [25]:
# Summary table
summary_df = pd.DataFrame([
    clean_on_clean,
    clean_on_adv,
    clean_on_combined,
    adv_trained_on_adv,
    adv_trained_on_clean,
    adv_trained_on_combined,
])

display(summary_df)


,model_eval,acc,f1
0,clean_model_on_clean_test,0.898824,0.748538
1,clean_model_on_adv_test,0.322353,0.052632
2,clean_model_on_combined_test,0.610588,0.303158
3,adv_trained_model_on_adv_test,0.877647,0.682927
4,adv_trained_model_on_clean_test,0.847059,0.601227
5,adv_trained_model_on_combined_test,0.862353,0.642202


In [26]:
# Save metrics
summary_path = RESULTS_DIR / "svm_deepfool_pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved: {summary_path}")


Saved: results\svm_deepfool_pipeline_summary.csv


## Dual-Stream Consistency Gate

This section mirrors the dual-stream detector structure from the LR and NN DeepFool notebooks.

### Gate logic

- **Nominal model** = clean model
- **Guardian model** = adversarially trained model
- Flag likely attacks using:
  1. prediction disagreement
  2. high-confidence disagreement
  3. nominal predicts benign while guardian predicts anomaly by a large enough margin

The gate is evaluated both as:

- a **final prediction system** for clean, adversarial, and combined inputs
- an **attack detector** using **FPR**, **TPR**, and **F1**


In [27]:
CONFIDENCE_THRESHOLD = 0.60
DISAGREEMENT_THRESHOLD = 0.55

def predict_with_confidence(model, X: np.ndarray):
    probs = predict_proba(model, X)
    preds = np.argmax(probs, axis=1)
    return preds, probs


class DualStreamDetector:
    def __init__(self, nominal_model, guardian_model):
        self.nominal_model = nominal_model
        self.guardian_model = guardian_model

    def detect_attacks(
        self,
        X: np.ndarray,
        confidence_threshold: float = CONFIDENCE_THRESHOLD,
        disagreement_threshold: float = DISAGREEMENT_THRESHOLD,
    ):
        preds_nominal, probs_nominal = predict_with_confidence(self.nominal_model, X)
        preds_guardian, probs_guardian = predict_with_confidence(self.guardian_model, X)

        n_samples = len(X)
        flags = np.zeros(n_samples, dtype=int)
        details = []

        for i in range(n_samples):
            yA = int(preds_nominal[i])
            yB = int(preds_guardian[i])
            pA = probs_nominal[i]
            pB = probs_guardian[i]

            conf_nominal = float(pA[yA])
            conf_guardian = float(pB[yB])

            detected = False
            reasons = []

            if yA != yB:
                detected = True
                reasons.append("disagreement")

            if yA == 0 and yB == 1:
                prob_diff = float(pB[1] - pA[1])
                if conf_nominal >= confidence_threshold and conf_guardian >= confidence_threshold:
                    if prob_diff >= disagreement_threshold:
                        detected = True
                        reasons.append("high_confidence_disagreement")
            else:
                prob_diff = float(pB[1] - pA[1])

            flags[i] = int(detected)
            details.append({
                "nominal_pred": yA,
                "guardian_pred": yB,
                "nominal_conf": conf_nominal,
                "guardian_conf": conf_guardian,
                "nominal_anom_prob": float(pA[1]),
                "guardian_anom_prob": float(pB[1]),
                "prob_diff_anomaly": prob_diff,
                "detected": bool(detected),
                "reason": ",".join(reasons) if reasons else "none",
            })

        final_preds = np.where(flags == 1, preds_guardian, preds_nominal)
        details_df = pd.DataFrame(details)
        return final_preds, flags, details_df


def evaluate_dual_stream_predictions(y_true: np.ndarray, y_pred: np.ndarray, name: str):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n[{name}] acc={acc:.4f} f1={f1:.4f}")
    print("confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    return {
        "model_eval": name,
        "acc": acc,
        "f1": f1,
    }


detector = DualStreamDetector(
    nominal_model=clean_model,
    guardian_model=adv_model,
)

gated_clean_pred, flags_clean, gate_clean_df = detector.detect_attacks(X_test)
gated_adv_pred, flags_adv, gate_adv_df = detector.detect_attacks(X_test_adv)
gated_combined_pred, flags_combined, gate_combined_df = detector.detect_attacks(X_test_combined)

gated_clean_metrics = evaluate_dual_stream_predictions(
    y_test, gated_clean_pred, "dual_stream_final_predictions_on_clean_test"
)
gated_adv_metrics = evaluate_dual_stream_predictions(
    y_test_adv, gated_adv_pred, "dual_stream_final_predictions_on_adv_test"
)
gated_combined_metrics = evaluate_dual_stream_predictions(
    y_test_combined, gated_combined_pred, "dual_stream_final_predictions_on_combined_test"
)

dual_stream_prediction_summary_df = pd.DataFrame([
    gated_clean_metrics,
    gated_adv_metrics,
    gated_combined_metrics,
])

print("\nDual-stream final prediction summary:")
display(dual_stream_prediction_summary_df)

y_attack_true = np.concatenate([
    np.zeros(len(flags_clean), dtype=int),
    np.ones(len(flags_adv), dtype=int),
])
y_attack_pred = np.concatenate([flags_clean, flags_adv])

dual_stream_detection_results = pd.DataFrame([{
    "attack": "DeepFool",
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "disagreement_threshold": DISAGREEMENT_THRESHOLD,
    "FPR": float(flags_clean.mean()),
    "TPR": float(flags_adv.mean()),
    "F1": float(f1_score(y_attack_true, y_attack_pred, zero_division=0)),
    "clean_model_acc_on_adv": float(clean_on_adv["acc"]),
    "guardian_model_acc_on_clean": float(adv_trained_on_clean["acc"]),
    "guardian_model_acc_on_adv": float(adv_trained_on_adv["acc"]),
}])

print("\nDual-stream attack-detection summary:")
display(dual_stream_detection_results)

print("\nGate reason counts on clean test:")
display(gate_clean_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))

print("\nGate reason counts on adversarial test:")
display(gate_adv_df["reason"].value_counts(dropna=False).rename_axis("reason").reset_index(name="count"))


def print_prediction_distribution(name: str, y_true: np.ndarray, y_pred: np.ndarray):
    true_dist = dict(zip(*np.unique(y_true.astype(int), return_counts=True)))
    pred_dist = dict(zip(*np.unique(y_pred.astype(int), return_counts=True)))
    print(f"{name} true label dist: {true_dist}")
    print(f"{name} predicted label dist: {pred_dist}")

print("\nPrediction distribution checks:")
print_prediction_distribution("clean gated", y_test, gated_clean_pred)
print_prediction_distribution("adv gated", y_test_adv, gated_adv_pred)
print_prediction_distribution("combined gated", y_test_combined, gated_combined_pred)



[dual_stream_final_predictions_on_clean_test] acc=0.7812 f1=0.5463
confusion matrix:
[[276  62]
 [ 31  56]]
              precision    recall  f1-score   support

           0     0.8990    0.8166    0.8558       338
           1     0.4746    0.6437    0.5463        87

    accuracy                         0.7812       425
   macro avg     0.6868    0.7301    0.7011       425
weighted avg     0.8121    0.7812    0.7925       425


[dual_stream_final_predictions_on_adv_test] acc=0.8282 f1=0.6332
confusion matrix:
[[289  49]
 [ 24  63]]
              precision    recall  f1-score   support

           0     0.9233    0.8550    0.8879       338
           1     0.5625    0.7241    0.6332        87

    accuracy                         0.8282       425
   macro avg     0.7429    0.7896    0.7605       425
weighted avg     0.8495    0.8282    0.8357       425


[dual_stream_final_predictions_on_combined_test] acc=0.8047 f1=0.5891
confusion matrix:
[[565 111]
 [ 55 119]]
              prec

,model_eval,acc,f1
0,dual_stream_final_predictions_on_clean_test,0.781176,0.546341
1,dual_stream_final_predictions_on_adv_test,0.828235,0.633166
2,dual_stream_final_predictions_on_combined_test,0.804706,0.589109



Dual-stream attack-detection summary:


,attack,confidence_threshold,disagreement_threshold,FPR,TPR,F1,clean_model_acc_on_adv,guardian_model_acc_on_clean,guardian_model_acc_on_adv
0,DeepFool,0.6,0.55,0.204706,0.696471,0.732673,0.322353,0.847059,0.877647



Gate reason counts on clean test:


,reason,count
0,none,338
1,disagreement,80
2,"disagreement,high_confidence_disagreement",7



Gate reason counts on adversarial test:


,reason,count
0,disagreement,241
1,none,129
2,"disagreement,high_confidence_disagreement",55



Prediction distribution checks:
clean gated true label dist: {np.int64(0): np.int64(338), np.int64(1): np.int64(87)}
clean gated predicted label dist: {np.int64(0): np.int64(307), np.int64(1): np.int64(118)}
adv gated true label dist: {np.int64(0): np.int64(338), np.int64(1): np.int64(87)}
adv gated predicted label dist: {np.int64(0): np.int64(313), np.int64(1): np.int64(112)}
combined gated true label dist: {np.int64(0): np.int64(676), np.int64(1): np.int64(174)}
combined gated predicted label dist: {np.int64(0): np.int64(620), np.int64(1): np.int64(230)}


In [28]:
# Save dual-stream outputs
dual_stream_prediction_summary_path = RESULTS_DIR / "svm_dual_stream_prediction_summary.csv"
dual_stream_detection_path = RESULTS_DIR / "svm_dual_stream_detection_results.csv"

dual_stream_prediction_summary_df.to_csv(dual_stream_prediction_summary_path, index=False)
dual_stream_detection_results.to_csv(dual_stream_detection_path, index=False)

gate_clean_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_clean_details.csv", index=False)
gate_adv_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_adv_details.csv", index=False)
gate_combined_df.to_csv(RESULTS_DIR / "svm_dual_stream_gate_combined_details.csv", index=False)

print(f"Saved: {dual_stream_prediction_summary_path}")
print(f"Saved: {dual_stream_detection_path}")


Saved: results\svm_dual_stream_prediction_summary.csv
Saved: results\svm_dual_stream_detection_results.csv
